# 11 — Expériences CNN 2D sur scalogrammes

L'objectif est une classification binaire plusieurs-vers-un : chaque tenseur temps-fréquence conduit à un logit brut. `BCEWithLogitsLoss` applique de façon stable la sigmoïde et l'entropie croisée. La pondération éventuelle et le seuil sont choisis sur validation uniquement.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import time
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import *
from src.helpers import count_parameters, set_seed

set_seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Périphérique sélectionné :", device)
print("CUDA disponible :", torch.cuda.is_available())

Périphérique sélectionné : cuda
CUDA disponible : True


## Corrélation croisée 2D manuelle

PyTorch nomme `Conv2d` une opération qui réalise en pratique une corrélation croisée. Pour une entrée \(H\times W\), un noyau \(K\), un padding \(P\) et un pas \(S\),

\[
H_{out}=\left\lfloor\frac{H+2P-K}{S}\right\rfloor+1,\quad
W_{out}=\left\lfloor\frac{W+2P-K}{S}\right\rfloor+1.
\]

Ainsi, pour 64 × 64, noyau 3 et pas 1 : sans padding on obtient 62 × 62, avec padding 1 on conserve 64 × 64.

In [2]:
import torch.nn.functional as F
from src.manual_ops import corr2d, manual_avg_pool2d, manual_max_pool2d

input_matrix = torch.tensor([[1., 2., 3.], [4., 5., 6.], [7., 8., 9.]])
kernel = torch.tensor([[1., 0.], [0., -1.]])
manual_output = corr2d(input_matrix, kernel, stride=1, padding=0)
pytorch_output = F.conv2d(input_matrix[None, None], kernel[None, None]).squeeze()
print("Entrée :\n", input_matrix, "\nNoyau :\n", kernel, "\nSortie :\n", manual_output)
print("Dimension : floor((3 + 2*0 - 2) / 1) + 1 = 2")
torch.testing.assert_close(manual_output, pytorch_output)

Entrée :
 tensor([[1., 2., 3.],
        [4., 5., 6.],
        [7., 8., 9.]]) 
Noyau :
 tensor([[ 1.,  0.],
        [ 0., -1.]]) 
Sortie :
 tensor([[-4., -4.],
        [-4., -4.]])
Dimension : floor((3 + 2*0 - 2) / 1) + 1 = 2


## Max-pooling et average-pooling manuels

In [3]:
pool_input = torch.arange(1, 17, dtype=torch.float32).reshape(4, 4)
manual_max = manual_max_pool2d(pool_input, kernel_size=2, stride=2)
manual_avg = manual_avg_pool2d(pool_input, kernel_size=2, stride=2)
torch.testing.assert_close(manual_max, F.max_pool2d(pool_input[None, None], 2, 2).squeeze())
torch.testing.assert_close(manual_avg, F.avg_pool2d(pool_input[None, None], 2, 2).squeeze())
print(pool_input, "\nMax :", manual_max, "\nMoyenne :", manual_avg)
assert (64 - 3) // 1 + 1 == 62
assert (64 + 2 - 3) // 1 + 1 == 64

tensor([[ 1.,  2.,  3.,  4.],
        [ 5.,  6.,  7.,  8.],
        [ 9., 10., 11., 12.],
        [13., 14., 15., 16.]]) 
Max : tensor([[ 6.,  8.],
        [14., 16.]]) 
Moyenne : tensor([[ 3.5000,  5.5000],
        [11.5000, 13.5000]])


## Modèles réutilisables et nombre de paramètres

In [4]:
from src.models import WESADScalogramCNN

cnn2d = WESADScalogramCNN().to(device)
toy = torch.randn(4, 3, 64, 64, device=device)
assert cnn2d(toy).shape == (4, 1)
print("CNN 2D :", count_parameters(cnn2d), "paramètres")
print("Périphérique modèle :", next(cnn2d.parameters()).device)
print("Périphérique lot :", toy.device)
assert next(cnn2d.parameters()).device == toy.device

CNN 2D : 89473 paramètres
Périphérique modèle : cuda:0
Périphérique lot : cuda:0


Le CNN emploie des champs récepteurs locaux et partage ses poids sur les deux dimensions temps-fréquence. Le nombre de paramètres accompagne les mesures de performance.

## Chargement du jeu de scalogrammes

In [5]:
from src.scalograms import WESADScalogramDataset, validate_subject_splits

scalogram_dir = PROJECT_ROOT / "data" / "processed" / "scalograms"
metadata_dir = PROJECT_ROOT / "data" / "processed" / "metadata"
validate_subject_splits()

def load_scalogram_datasets():
    return tuple(
        WESADScalogramDataset(
            scalogram_dir / f"X_{split}.pt",
            scalogram_dir / f"y_{split}.pt",
            metadata_dir / f"windows_{split}.csv",
        )
        for split in ("train", "validation", "test")
    )

if (scalogram_dir / "X_train.pt").exists():
    datasets = load_scalogram_datasets()
    sample_x, sample_y, sample_metadata = datasets[0][0]
    print(sample_x.shape, sample_y.shape, sample_metadata["subject_id"])
else:
    datasets = None
    print("Scalogrammes absents : exécuter le notebook 10.")

torch.Size([3, 64, 64]) torch.Size([]) S3


## Protocole d'entraînement

Chaque variante repart de la graine 42. Adam, le taux d'apprentissage, la régularisation, la taille de lot, le nombre maximal d'époques et la patience viennent de `src.config`. Si la pondération est comparée, `pos_weight` est calculé uniquement sur les étiquettes d'entraînement. La variante et le seuil (0,10 à 0,90) maximisant le macro-F1 sont déterminés sur validation. Le test n'est chargé pour l'inférence qu'après gel de ces choix.

In [6]:
from src.experiments import run_validation_selected_experiment

RUN_CNN2D_TRAINING = False
results = {}
if RUN_CNN2D_TRAINING:
    if datasets is None:
        raise FileNotFoundError("Exécuter le notebook 10 avant l'entraînement.")
    validation_metadata = pd.read_csv(metadata_dir / "windows_validation.csv")
    test_metadata = pd.read_csv(metadata_dir / "windows_test.csv")
    common = {
        "seed": RANDOM_SEED,
        "subject_split": SPLIT_SUBJECTS,
        "input_channels": SCALOGRAM_CHANNELS,
        "input_shape": [3, 64, 64],
        "normalization_statistics": "artifacts/preprocessing/scalograms/scalogram_metadata.json",
    }
    results["cnn2d"] = run_validation_selected_experiment(
        WESADScalogramCNN,
        datasets,
        validation_metadata,
        test_metadata,
        PROJECT_ROOT / "artifacts" / "models" / "cnn2d",
        {**common, "model_class": "WESADScalogramCNN", "architecture": "16-32-64"},
        device,
        compare_weighted_loss=True,
    )
else:
    print("Non exécuté : activer RUN_CNN2D_TRAINING pour les données réelles.")

Non exécuté : activer RUN_CNN2D_TRAINING pour les données réelles.


## Métriques de validation et de test

In [7]:
cnn2d_artifact_dir = PROJECT_ROOT / "artifacts" / "models" / "cnn2d"
metric_files = {
    "Validation": cnn2d_artifact_dir / "validation_metrics.json",
    "Test final": cnn2d_artifact_dir / "test_metrics.json",
    "Résumé entraînement": cnn2d_artifact_dir / "training_summary.json",
}
if all(path.exists() for path in metric_files.values()):
    loaded_metrics = {}
    for section, path in metric_files.items():
        with path.open(encoding="utf-8") as handle:
            loaded_metrics[section] = json.load(handle)
    metric_names = ["macro_f1", "weighted_f1", "stress_precision", "stress_recall", "roc_auc", "average_precision"]
    metrics_table = pd.DataFrame({
        section: {name: values.get(name) for name in metric_names}
        for section, values in loaded_metrics.items()
        if section != "Résumé entraînement"
    })
    display(metrics_table)
    display(pd.Series(loaded_metrics["Résumé entraînement"], name="entraînement"))
    display(pd.read_csv(cnn2d_artifact_dir / "per_subject_metrics.csv"))
else:
    print("Métriques absentes : terminer d'abord la cellule d'entraînement CNN 2D.")

,Validation,Test final
macro_f1,0.862566,0.843993
weighted_f1,0.884824,0.874975
stress_precision,0.811765,0.953488
stress_recall,0.802326,0.640625
roc_auc,0.939489,0.973023
average_precision,0.875823,0.939067


best_epoch                       10
best_validation_loss       0.519043
epochs_trained                   20
early_stopped                  True
training_time_seconds      4.623403
gradient_clip_max_norm         None
gradient_norms_recorded       False
numerically_stable             True
numerical_failure              None
inference_time_seconds      0.03525
Name: entraînement, dtype: object

,subject_id,n_windows,macro_f1,weighted_f1,non_stress_precision,non_stress_recall,stress_precision,stress_recall,confusion_matrix,roc_auc,average_precision
0,S11,144,0.983636,0.986111,0.990000,0.990000,0.977273,0.977273,"[[99, 1], [1, 43]]",0.999091,0.998106
1,S14,144,0.684211,0.755848,0.773438,0.990000,0.937500,0.340909,"[[99, 1], [29, 15]]",0.942955,0.875022
2,S2,138,0.820779,0.860079,0.857143,0.979592,0.923077,0.600000,"[[96, 2], [16, 24]]",0.977551,0.886202


## Sauvegarde par `state_dict` et contrôle du rechargement

In [8]:
checkpoint_path = PROJECT_ROOT / "artifacts/models/cnn2d/best_model.pt"
if datasets is not None and checkpoint_path.exists():
    batch = next(iter(torch.utils.data.DataLoader(datasets[2], batch_size=4)))[0].to(device)
    trained = WESADScalogramCNN().to(device)
    trained.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
    trained.eval()
    with torch.no_grad():
        logits_before = trained(batch)
    reloaded = WESADScalogramCNN().to(device)
    reloaded.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
    reloaded.eval()
    with torch.no_grad():
        logits_after = reloaded(batch)
    torch.testing.assert_close(logits_before, logits_after)
    print("Logits identiques après rechargement.")

Logits identiques après rechargement.
